# ML-04 — Data Contract: Refresh / Content Opportunity Scoring

This notebook maps my **Refresh / Content Opportunity Scoring** lane onto the FlyRank warehouse.

**Development month:** March 2026 (`month=2026-03`).  
**Decision moment:** end of 2026-03-21.  
**Feature window:** 2026-03-01 through 2026-03-21.  
**Forward proxy window:** 2026-03-22 through 2026-03-31.

The final month (June 2026) remains sealed and is not used to develop the label logic.

> This notebook reads the gated warehouse directly from Hugging Face. In Colab, add a Secret named `HF_TOKEN` containing a plain READ token. The token is never printed or stored in this notebook.


## 1. Contract in plain words

1. **One row means:** one pseudonymized content item (`client_hash_id` + `content_hash_id`) summarized at the decision moment.
2. **Tables:** `fact_content_daily_performance` for daily Search Console / Analytics measurements. `dim_content` is available later for safe content metadata, but the five-feature frame below deliberately stays on the daily fact.
3. **Time window:** March 2026 is the development month. Features use March 1–21; the provisional forward proxy uses March 22–31. June 2026 is not touched.
4. **What I would rank:** pages by review priority. For this exercise the binary proxy is **future search-impression decline**: the page had at least 100 impressions in March 12–21 and impressions in March 22–31 are at least 20% lower than March 12–21.
5. **One deliberate exclusion:** any measurement from March 22–31 is excluded from features because it belongs to the outcome window. IDs are context only, never model features.

The output is decision support: a ranked queue for an editor to review. A positive proxy does not prove that editing a page will cause recovery.


In [2]:
# Secure warehouse setup — no token is hard-coded or printed.
import os
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN is missing. In Colab: Secrets (key icon) → add HF_TOKEN → "
        "enable notebook access → Runtime > Run all."
    )

con = duckdb.connect()
safe_token = HF_TOKEN.replace("'", "''")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{safe_token}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = (
    f"read_parquet('{REL}/fact_content_daily_performance/"
    "month=2026-03/*.parquet', hive_partitioning=true)"
)

print("Warehouse connection ready. Development partition: 2026-03")


Warehouse connection ready. Development partition: 2026-03


## 2. Field roles

**Features — exactly five, all knowable by the end of March 21**

- `imp_feature_21d` — knowable at the decision moment because it sums GSC impressions observed March 1–21.
- `clicks_feature_21d` — knowable because it sums clicks already observed March 1–21.
- `ctr_feature_21d` — knowable because it is computed only from those observed clicks and impressions.
- `avg_position_feature_21d` — knowable because it averages GSC position only through March 21.
- `active_days_feature_21d` — knowable because it counts pre-decision days with impressions.

**Label / proxy**

- `decline_proxy` — computed from the *future* March 22–31 outcome versus March 12–21. It is never a feature.

**Context**

- `client_hash_id`, `content_hash_id` — pseudonymous grouping/join/split keys only.

**Excluded**

- March 22–31 GSC/GA4 measurements — future information at the decision moment.
- `gsc_data_available` / `ga4_data_available` — availability controls, not behavioral predictors here.
- Product decisions/scores and any label-derived flag — leakage risk.
- June 2026 `_sample` / final-month data — sealed test month.


In [3]:
# The three verification queries are kept as literal SQL strings so the contract is auditable.
# They are executed in Section 3.

GRAIN_QUERY = f'''
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
FROM {MARCH}
GROUP BY 1,2,3
HAVING COUNT(*) > 1
LIMIT 10
'''

COUNT_SPAN_QUERY = f'''
SELECT
    COUNT(*) AS daily_rows,
    COUNT(DISTINCT client_hash_id || '|' || content_hash_id) AS content_items,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM {MARCH}
'''

AVAILABILITY_QUERY = f'''
SELECT
    COUNT(*) AS march_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE
    ) AS both_gsc_ga4_available_rows
FROM {MARCH}
'''

print("Prepared exactly three verification queries: grain, count/date span, availability.")


Prepared exactly three verification queries: grain, count/date span, availability.


## 3. Three verification queries, five-feature frame, and the leakage trap

The next cell executes **exactly three contract-verification queries**:

1. **Grain:** duplicate `(report_date, client_hash_id, content_hash_id)` rows. Zero returned rows supports the documented daily grain.
2. **Slice count + date span:** number of March rows/content items and `MIN/MAX(report_date)`.
3. **Availability:** uses `IS TRUE` explicitly. This matters because availability flags can be three-valued; `NULL` must not silently become usable data.

After those checks, I build one content-level frame from the same March partition. The frame uses only GSC-available rows. I require at least 100 impressions in March 12–21 before defining the decline proxy so tiny-volume movement is less likely to dominate.


In [4]:
# --- Verification query 1: grain ---
grain_check = con.sql(GRAIN_QUERY).df()
print("QUERY 1 — duplicate grain rows (expected: 0 rows)")
display(grain_check)

# --- Verification query 2: slice count + date span ---
count_span = con.sql(COUNT_SPAN_QUERY).df()
print("\nQUERY 2 — March slice row count and date span")
display(count_span)

# --- Verification query 3: availability, deliberately using IS TRUE ---
availability = con.sql(AVAILABILITY_QUERY).df()
print("\nQUERY 3 — availability")
display(availability)

# Build the five-feature frame + forward proxy.
# This is feature engineering, not an additional contract-verification query.
frame_sql = f'''
WITH usable AS (
    SELECT *
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
),
page_window AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
                 THEN gsc_impressions ELSE 0 END) AS imp_feature_21d,
        SUM(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
                 THEN gsc_clicks ELSE 0 END) AS clicks_feature_21d,
        AVG(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
                 AND gsc_avg_position > 0
                 THEN gsc_avg_position END) AS avg_position_feature_21d,
        COUNT(DISTINCT CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
                             AND gsc_impressions > 0
                            THEN report_date END) AS active_days_feature_21d,

        SUM(CASE WHEN report_date BETWEEN DATE '2026-03-12' AND DATE '2026-03-21'
                 THEN gsc_impressions ELSE 0 END) AS prior10_impressions,
        SUM(CASE WHEN report_date BETWEEN DATE '2026-03-22' AND DATE '2026-03-31'
                 THEN gsc_impressions ELSE 0 END) AS future10_impressions
    FROM usable
    GROUP BY 1,2
),
labeled AS (
    SELECT *,
        100.0 * clicks_feature_21d / NULLIF(imp_feature_21d, 0) AS ctr_feature_21d,
        CASE
            WHEN prior10_impressions >= 100
             AND future10_impressions <= 0.80 * prior10_impressions
            THEN 1 ELSE 0
        END AS decline_proxy
    FROM page_window
    WHERE prior10_impressions >= 100
)
SELECT * FROM labeled
'''

frame = con.sql(frame_sql).df()

feature_cols = [
    "imp_feature_21d",
    "clicks_feature_21d",
    "ctr_feature_21d",
    "avg_position_feature_21d",
    "active_days_feature_21d",
]
context_cols = ["client_hash_id", "content_hash_id"]

print(f"\nFeature-frame rows: {len(frame):,}")
print(f"Observed decline-proxy rate: {frame['decline_proxy'].mean():.1%}")
display(frame[context_cols + feature_cols + ["decline_proxy"]].head(10))

# --- The trap: deliberately leak the label, then remove it ---
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

y = frame["decline_proxy"].astype(int)
groups = frame["client_hash_id"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(frame, y, groups=groups))

def quick_auc(cols):
    model = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(max_iter=1000, class_weight="balanced")
    )
    model.fit(frame.iloc[train_idx][cols], y.iloc[train_idx])
    p = model.predict_proba(frame.iloc[test_idx][cols])[:, 1]
    return roc_auc_score(y.iloc[test_idx], p)

honest_auc = quick_auc(feature_cols)

# DELIBERATE LEAK: a direct copy of the label.
frame["label_leak_DO_NOT_USE"] = frame["decline_proxy"]
leaky_auc = quick_auc(feature_cols + ["label_leak_DO_NOT_USE"])

print(f"Honest five-feature held-out AUROC: {honest_auc:.3f}")
print(f"With deliberate label leak AUROC: {leaky_auc:.3f}")

# Delete the trap and keep only the honest frame.
frame = frame.drop(columns=["label_leak_DO_NOT_USE"])
assert "label_leak_DO_NOT_USE" not in frame.columns
print("Leak column deleted. Honest feature count:", len(feature_cols))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

QUERY 1 — duplicate grain rows (expected: 0 rows)


,report_date,client_hash_id,content_hash_id,n



QUERY 2 — March slice row count and date span


,daily_rows,content_items,min_date,max_date
0,9841378,331437,2026-03-01,2026-03-31


...

,march_rows,gsc_available_rows,both_gsc_ga4_available_rows
0,9841378,3611061,364347


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Feature-frame rows: 71,777
Observed decline-proxy rate: 23.4%


,client_hash_id,content_hash_id,imp_feature_21d,clicks_feature_21d,ctr_feature_21d,avg_position_feature_21d,active_days_feature_21d,decline_proxy
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,5229.0,6.0,0.114745,7.128867,21,1
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,313.0,0.0,0.000000,3.735151,21,0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,4545.0,3.0,0.066007,6.567922,21,1
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,3495.0,12.0,0.343348,7.270357,21,0
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,350.0,1.0,0.285714,3.599351,21,1
5,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,4561.0,17.0,0.372725,5.631797,21,0
6,client_73cda7b4e4f265ea,content_20403327d8d9374c,2569.0,7.0,0.272480,7.964566,21,1
7,client_73cda7b4e4f265ea,content_f8df6b20d18c4374,685.0,1.0,0.145985,7.326633,21,1
8,client_73cda7b4e4f265ea,content_419dc7d89aee9698,1515.0,3.0,0.198020,5.221440,21,1
9,client_73cda7b4e4f265ea,content_30fc0ffeed8d67e6,1735.0,15.0,0.864553,8.647642,21,1


Honest five-feature held-out AUROC: 0.590
With deliberate label leak AUROC: 1.000
Leak column deleted. Honest feature count: 5


## 4. Data limitation

A named limitation is the warehouse's **unbalanced history and source availability**. Different clients begin GSC/GA4 tracking on different dates, and GA4/GSC availability flags can be `TRUE`, `FALSE`, or `NULL`. Therefore, a zero before tracking begins cannot be interpreted as zero user activity.

This notebook reduces that risk by developing on a mid-panel month and filtering GSC measurements with `gsc_data_available IS TRUE`. However, the resulting March slice represents only content with usable GSC data and enough prior-window impressions. Findings may not generalize to low-volume pages, newly tracked clients, or pages without usable search history.

The forward-decline proxy is also observational. It measures a directional change in search impressions; it does not tell us why the change happened or whether a refresh would cause recovery.


In [5]:
# Final audit of the artifact itself.
assert len(feature_cols) == 5
assert "label_leak_DO_NOT_USE" not in frame.columns
assert frame["decline_proxy"].isin([0, 1]).all()
assert pd.to_datetime(count_span.loc[0, "min_date"]).strftime("%Y-%m") == "2026-03"
assert pd.to_datetime(count_span.loc[0, "max_date"]).strftime("%Y-%m") == "2026-03"

print("Final checks passed:")
print("- exactly five honest features")
print("- deliberate leak removed")
print("- binary proxy present")
print("- development data stays in March 2026")


Final checks passed:
- exactly five honest features
- deliberate leak removed
- binary proxy present
- development data stays in March 2026


## 5. Self-check

- [x] Five plain-words contract answers are stated.
- [x] Exactly three verification queries are defined: grain, row count/date span, and availability.
- [x] Availability is checked with `IS TRUE`.
- [x] The feature frame contains exactly five features.
- [x] Every feature has an “available when?” explanation.
- [x] One label-derived column is deliberately added, its score is compared, and the column is deleted.
- [x] The final retained score is the honest five-feature score.
- [x] One limitation of the slice is named.
- [x] June 2026 / `_sample` is not used for label development.
- [x] No client names, URLs, private queries, or Hugging Face token are stored in the notebook.
- [ ] Run all cells in Colab with the `HF_TOKEN` Secret so the three real warehouse outputs and leakage scores are embedded.
- [ ] Commit the executed notebook as `work/notebooks/w03_data_contract.ipynb`, then submit the public repo URL.
